# Lista 10

## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

## Zadanie 1

In [2]:
data = pd.read_csv('./data/pzz_fleet_data_full.csv')

In [3]:
data

,bus_id,temperature,vibration,speed,fuel_consumption,load,outlier_type,is_failure,timestamp
0,30,65.56,18.26,41.10,38.06,50.36,normal,0,2024-01-21 01:42:00
1,2,74.02,25.24,32.78,33.07,62.53,normal,0,2024-05-30 18:04:00
2,29,81.81,20.31,30.07,32.91,28.47,normal,0,2024-11-16 03:47:00
3,74,75.90,14.90,32.42,40.43,39.40,normal,0,2024-10-08 08:08:00
4,98,65.98,11.97,19.71,30.07,25.19,normal,0,2024-03-26 10:03:00
...,...,...,...,...,...,...,...,...,...
9995,57,75.87,16.70,30.96,36.94,87.95,normal,0,2024-10-19 18:51:00
9996,96,75.23,15.79,35.95,29.14,62.89,normal,0,2024-05-12 08:16:00
9997,14,69.84,11.57,57.43,31.79,29.89,normal,0,2024-02-06 18:15:00
9998,51,76.55,10.58,33.85,38.96,63.55,normal,0,2024-09-29 20:12:00


In [4]:
data2 = pd.read_json('./data/pzz_fleet_physical_limits.json')

In [5]:
data2

,temperature,vibration,speed,fuel_consumption,load
0,0,0,0,0,0
1,200,100,150,100,100


### Podpunkt a

In [6]:
data.iloc[:, 1:6]

,temperature,vibration,speed,fuel_consumption,load
0,65.56,18.26,41.10,38.06,50.36
1,74.02,25.24,32.78,33.07,62.53
2,81.81,20.31,30.07,32.91,28.47
3,75.90,14.90,32.42,40.43,39.40
4,65.98,11.97,19.71,30.07,25.19
...,...,...,...,...,...
9995,75.87,16.70,30.96,36.94,87.95
9996,75.23,15.79,35.95,29.14,62.89
9997,69.84,11.57,57.43,31.79,29.89
9998,76.55,10.58,33.85,38.96,63.55


In [7]:
def iqr_row_outlier(data):
    iqr = np.quantile(data, 0.75) - np.quantile(data, 0.25)
    return np.logical_or(data > np.quantile(data, 0.75) + 1.5 * iqr, data < np.quantile(data, 0.25) - 1.5 * iqr)

def iqr_outliers(data):
    res = np.zeros_like(data.iloc[:, 0])
    for col in data.columns[1:6]:
        res = np.logical_or(res, iqr_row_outlier(data[col]))
    return res

In [8]:
np.sum(iqr_outliers(data))

np.int64(484)

In [9]:
def zscore_row_outlier(data):
    mean = np.mean(data)
    std= np.std(data)
    z = (data - mean) / std
    return np.abs(z) > 3

def zscore_outliers(data):
    res = np.zeros_like(data.iloc[:, 0])
    for col in data.columns[1:6]:
        res = np.logical_or(res, zscore_row_outlier(data[col]))
    return res

In [10]:
np.sum(zscore_outliers(data))

np.int64(210)

In [11]:
model = IsolationForest(contamination=0.03, random_state=42)
preds = model.fit_predict(data.iloc[:, 1:6])

In [12]:
# IsolationForest zwraca 1 dla normalnych, -1 dla anomalii
np.sum(preds == -1)

np.int64(300)

In [13]:
iqr_detected_outliers = np.sum(np.logical_and(iqr_outliers(data), data['outlier_type'] != 'normal'))
zscore_detected_outliers = np.sum(np.logical_and(zscore_outliers(data), data['outlier_type'] != 'normal'))
isolation_forest_detected_outliers = np.sum(np.logical_and(preds == -1, data['outlier_type'] != 'normal'))

In [14]:
print(f"IQR wykrywa: {iqr_detected_outliers}")
print(f"Zscore wykrywa: {zscore_detected_outliers}")
print(f"Isolation forest wykrywa: {isolation_forest_detected_outliers}")

IQR wykrywa: 300
Zscore wykrywa: 210
Isolation forest wykrywa: 295


In [15]:
combined_coverage = np.sum(np.logical_and(iqr_outliers(data), zscore_outliers(data), preds == -1))

In [16]:
print(f'Wspolnie pokrywaja: {combined_coverage}')

Wspolnie pokrywaja: 210


### Podpunkt b

In [17]:
def classify_outliers(data, data2, idx):
    return np.logical_or(data[idx].iloc[:, 1:6] < data2.iloc[0], data[idx].iloc[:, 1:6] > data2.iloc[1])

In [18]:
iqr_out = iqr_outliers(data)
zscore_out = zscore_outliers(data)
isolation_out = preds == -1

iqr_classified_outliers = classify_outliers(data, data2, iqr_out)
zscore_classified_outliers = classify_outliers(data, data2, zscore_out)
isolation_classified_outliers = classify_outliers(data, data2, isolation_out)

### Podpunkt c

In [19]:
outlier_indices = isolation_classified_outliers.index

In [20]:
outlier_indices

Index([  17,   20,   25,   67,  213,  257,  283,  414,  442,  507,
       ...
       9695, 9699, 9732, 9747, 9768, 9817, 9850, 9853, 9856, 9891],
      dtype='int64', length=300)

In [26]:
isolation_classified_outliers

,temperature,vibration,speed,fuel_consumption,load
17,False,False,False,False,False
20,True,True,True,True,True
25,False,False,False,False,False
67,True,True,True,True,True
213,True,True,True,True,True
...,...,...,...,...,...
9817,True,True,True,True,True
9850,True,True,True,True,True
9853,False,False,False,False,False
9856,True,True,True,True,True


In [21]:
data_strategy_1 = data.drop(index=outlier_indices)

In [25]:
lower_limit = data.iloc[:, 1:6].quantile(0.01)
upper_limit = data.iloc[:, 1:6].quantile(0.99)

data_strategy_2 = data.clip(lower=lower_limit, upper=upper_limit, axis=1)

In [27]:
data_strategy_3 = data.copy()

# Dodajemy nową cechę: 1 jeśli to outlier, 0 jeśli normalny odczyt
data_strategy_3['is_outlier'] = 0
data_strategy_3.loc[outlier_indices, 'is_outlier'] = 1

In [30]:
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

def evaluate_strategy(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds)

# Porównanie
f1_s1 = evaluate_strategy(data_strategy_1.loc[:, 1:6], data_strategy_1['is_failure'])
f1_s2 = evaluate_strategy(data_strategy_2.loc[:, 1:6], data_strategy_2['is_failure'])
f1_s3 = evaluate_strategy(data_strategy_3.loc[:, 1:6], data_strategy_3['is_failure'])

print(f"F1 Usunięcie: {f1_s1}")
print(f"F1 Winsoryzacja: {f1_s2}")
print(f"F1 Oznaczenie: {f1_s3}")

TypeError: cannot do slice indexing on Index with these indexers [1] of type int